In [1]:
# Import libraries used for data manipulation and machine learning

import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegressionCV
from sklearn.linear_model import LogisticRegression

In [2]:

# Load the simulated bundled payment dataset

data = pd.read_csv("~/Downloads/simulated_bundled_data.csv")

# Remove index column if it exists

if "Unnamed: 0" in data.columns:
    data = data.drop(columns=["Unnamed: 0"])

In [3]:
# Define a causal ordering of variables
# Each node will be regressed only on variables that occur earlier in the order

node_order = [
    "DME",
    "CL",
    "P",
    "H",
    "LTH",
    "PBD",
    "RF",
    "SNF",
    "HHA",
    "HOS",
    "HO",
    "OT",
    "BP"
]

In [4]:
# Recover the network using LASSO regression
# Each node is predicted using all earlier nodes
# LASSO shrinks weak coefficients to zero, helping identify the network structure

def fit_lasso_network(data, node_order, coef_threshold=0.05):

    parent_dict = {}
    lasso_details = {}

    for i, target in enumerate(node_order):

        candidate_parents = node_order[:i]

        if len(candidate_parents) == 0:
            parent_dict[target] = []
            lasso_details[target] = pd.Series(dtype=float)
            continue

        X = data[candidate_parents]
        y = data[target]

        model = LogisticRegressionCV(
            Cs=np.logspace(-3, 2, 20),
            penalty="l1",
            solver="liblinear",
            cv=5,
            scoring="neg_log_loss",
            max_iter=3000
        )

        model.fit(X, y)

        coefs = pd.Series(model.coef_[0], index=candidate_parents)

        non_zero = coefs[coefs != 0]

        selected = non_zero[non_zero.abs() >= coef_threshold]

        parent_dict[target] = list(selected.index)

        lasso_details[target] = selected.sort_values(
            key=np.abs,
            ascending=False
        )

    return parent_dict, lasso_details




In [5]:
# Run LASSO network recovery

parent_dict, lasso_details = fit_lasso_network(data, node_order)



# Print recovered network structure

print("\nRecovered network from LASSO")

for node in node_order:
    print(node, "<-", parent_dict[node])



Recovered network from LASSO
DME <- []
CL <- []
P <- ['DME', 'CL']
H <- ['DME', 'CL', 'P']
LTH <- ['P', 'H']
PBD <- []
RF <- ['P', 'H', 'LTH']
SNF <- ['P', 'H']
HHA <- ['P', 'RF']
HOS <- ['H', 'LTH', 'PBD']
HO <- ['LTH', 'PBD']
OT <- ['DME', 'CL', 'LTH', 'PBD', 'RF', 'SNF', 'HHA', 'HOS']
BP <- ['H', 'LTH', 'PBD', 'SNF', 'HHA', 'HOS', 'HO', 'OT']


In [7]:

# Fit logistic regression models for each node
# These models will be used as structural equations during simulation

def fit_structural_models(data, node_order, parent_dict):

    fitted_models = {}
    base_rates = {}

    for node in node_order:

        parents = parent_dict[node]

        base_rates[node] = data[node].mean()

        if len(parents) == 0:

            fitted_models[node] = None

        else:

            X = data[parents]
            y = data[node]

            model = LogisticRegression(
                penalty=None,
                solver="lbfgs",
                max_iter=3000
            )

            model.fit(X, y)

            fitted_models[node] = model

    return fitted_models, base_rates

# Estimate structural models

fitted_models, base_rates = fit_structural_models(data, node_order, parent_dict)


In [8]:
# Simulate the intervention do(H = value)
# This generates counterfactual outcomes using the recovered network

def simulate_do_H(data, node_order, parent_dict, fitted_models, h_value, n_rep=200, random_state=42):

    rng = np.random.default_rng(random_state)

    bp_means = []

    for rep in range(n_rep):

        sim = pd.DataFrame(index=data.index)

        for node in node_order:

            if len(parent_dict[node]) == 0:
                sim[node] = data[node].values

        sim["H"] = h_value

        for node in node_order:

            if node == "H":
                continue

            parents = parent_dict[node]

            if len(parents) == 0:
                continue

            model = fitted_models[node]

            probs = model.predict_proba(sim[parents])[:, 1]

            sim[node] = rng.binomial(1, probs)

        bp_means.append(sim["BP"].mean())

    return np.mean(bp_means)



In [9]:
# Estimate causal impact of H on BP

bp_do_h1 = simulate_do_H(data, node_order, parent_dict, fitted_models, h_value=1)

bp_do_h0 = simulate_do_H(data, node_order, parent_dict, fitted_models, h_value=0)

causal_effect = bp_do_h1 - bp_do_h0

In [10]:
# Display results

print("\nEstimated causal impact of H on BP")

print("P(BP = 1 | do(H = 1)) =", round(bp_do_h1,4))

print("P(BP = 1 | do(H = 0)) =", round(bp_do_h0,4))

print("Risk difference =", round(causal_effect,4))


Estimated causal impact of H on BP
P(BP = 1 | do(H = 1)) = 0.9812
P(BP = 1 | do(H = 0)) = 0.9299
Risk difference = 0.0513


In [11]:
# Estimate the effect conditional on P
# This reproduces the type of analysis shown in Netica

def simulate_do_H_fixed_P(data, node_order, parent_dict, fitted_models, h_value, p_value, n_rep=200):

    rng = np.random.default_rng()

    bp_means = []

    for rep in range(n_rep):

        sim = pd.DataFrame(index=data.index)

        for node in node_order:

            if len(parent_dict[node]) == 0:
                sim[node] = data[node].values

        sim["P"] = p_value
        sim["H"] = h_value

        for node in node_order:

            if node in ["P","H"]:
                continue

            parents = parent_dict[node]

            if len(parents) == 0:
                continue

            model = fitted_models[node]

            probs = model.predict_proba(sim[parents])[:,1]

            sim[node] = rng.binomial(1, probs)

        bp_means.append(sim["BP"].mean())

    return np.mean(bp_means)

In [12]:
# Conditional intervention results

bp_h1_p0 = simulate_do_H_fixed_P(data, node_order, parent_dict, fitted_models, 1, 0)

bp_h1_p1 = simulate_do_H_fixed_P(data, node_order, parent_dict, fitted_models, 1, 1)



print("\nConditional intervention results")

print("P(BP = 1 | do(H = 1), P = 0) =", round(bp_h1_p0,4))

print("P(BP = 1 | do(H = 1), P = 1) =", round(bp_h1_p1,4))


Conditional intervention results
P(BP = 1 | do(H = 1), P = 0) = 0.9769
P(BP = 1 | do(H = 1), P = 1) = 0.9853


In [ ]:
# First, the code recovers the causal network structure from the simulated data
# using repeated LASSO logistic regressions. Each node is predicted only from
# variables that occur earlier in the assumed causal ordering. The LASSO penalty
# shrinks weak relationships toward zero, leaving a sparse set of parent variables
# that approximates the original directed acyclic graph (DAG).

# Second, the code fits structural logistic regression models for each node using
# the recovered parent sets. These models act as the structural equations of the
# network and describe how each variable is generated from its parent variables.

# Third, the code estimates the causal impact of H on BP by performing a
# do-intervention simulation. The simulation sets H = 1 or H = 0, generates all
# downstream variables according to the recovered structural equations, and then
# computes the resulting probability that BP = 1.

# This produces the counterfactual quantities:
# P(BP = 1 | do(H = 1)) and P(BP = 1 | do(H = 0)).

# The difference between these probabilities represents the causal impact of H
# on BP, which corresponds to the intervention analysis obtained using Netica.